# Credit Portfolio Risk Intelligence
## Notebook 3 - XGBoost PD Model

### PACE: Construct - Model Upgrade

**Objective:** Extend the baseline logistic regression PD model 
with XGBoost gradient boosting to capture non-linear feature 
interactions and improve discriminatory power.

**Baseline:** Logistic Regression - Gini 42.8%, AUC 0.714  
**Expected improvement:** Gini 58-68% (Good range)

**Why XGBoost?:**
- Captures non-linear relationships between risk features
- Learns interaction effects (e.g. grade × DTI combinations)
- No assumption of linearity - better suited to complex 
  borrower behaviour patterns
- Trade-off: requires SHAP explainability layer for auditability

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)
print("Libraries loaded successfully")
print(f"XGBoost version: {__import__('xgboost').__version__}")

Libraries loaded successfully
XGBoost version: 3.2.0


In [15]:
# Load cleaned dataset
df = pd.read_csv('../data/loans_clean.csv')

# Recreate installment_to_income
median_income = df[df['annual_inc'] > 0]['annual_inc'].median()
df['annual_inc'] = df['annual_inc'].replace(0, median_income)
df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12)

print(f"Shape: {df.shape}")
print(f"Default rate: {df['default'].mean() * 100:.1f}%")

Shape: (44006, 85)
Default rate: 20.5%


In [16]:
# Same feature set as logistic regression v4
feature_cols = [
    'loan_amnt',
    'installment',
    'annual_inc',
    'int_rate',
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc',
    'installment_to_income',
    'fico_range_low'
]

target_col = 'default'

# Class imbalance ratio for scale_pos_weight

neg = (df[target_col] == 0).sum()
pos = (df[target_col] == 1).sum()
imbalance_ratio = round(neg/pos, 1)

print(f"Features: {len(feature_cols)}")
print(f"Non-defaults: {neg:,}")
print(f"Defaults: {pos:,}")
print(f"Imbalance ratio: {imbalance_ratio}")

# Split
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size = 0.2, random_state = 42, stratify = y
)

print(f"\nTraining set: {X_train.shape[0]:,}")
print(f"Test set: {X_test.shape[0]:,}")
print(f"Default rate - Train: {y_train.mean()*100:.1f}%")
print(f"Default rate - Test: {y_test.mean()*100:.1f}%")


Features: 14
Non-defaults: 34,978
Defaults: 9,028
Imbalance ratio: 3.9

Training set: 35,204
Test set: 8,802
Default rate - Train: 20.5%
Default rate - Test: 20.5%


In [17]:
# XGBoost Training (no scaling required)

# Using features directly
print(f"Training XGBoost model...")
print(f"Parameters:")
print(f"n_estimators: 300")
print(f"max_depth: 4")
print(f"learning_rate: 0.05")
print(f"scale_pos_weight: {imbalance_ratio}")
print()

model_xgb = XGBClassifier(
    n_estimators = 300, max_depth =4, learning_rate = 0.05,
    subsample = 0.8, colsample_bytree = 0.8, scale_post_weight = imbalance_ratio,
    reg_alpha = 0.1, reg_lambda = 1.0, random_state = 42, eval_metric = 'auc',
    verbosity = 0
)
model_xgb.fit(X_train,y_train)

print(f"Model Trained Successfully.")
print(f"Trees built: {model_xgb.n_estimators}")

Training XGBoost model...
Parameters:
n_estimators: 300
max_depth: 4
learning_rate: 0.05
scale_pos_weight: 3.9

Model Trained Successfully.
Trees built: 300


In [18]:
# Model Metric Evaluation
#Generate Predictions
y_pred_xgb = model_xgb.predict_proba(X_test)[:,1]

#Calculate metrics
auc_xgb = roc_auc_score(y_test, y_pred_xgb)
gini_xgb = (2*auc_xgb) - 1

print(f"XGBoost Performance:")
print("-"*40)
print(f"ROC AUC: {auc_xgb:,.4f}")
print(f"Gini Coefficient: {gini_xgb:.4f} ({gini_xgb*100:.1f}%)")

print(f"\nModel Comparison: ")
print("-"*40)
print(f"{'Model':<25} {'AUC':>8} {'Gini':>8}")
print("-"*40)
print(f"{'Logistic Regression':<25} {'0.7140':>8} {'42.8%':>8}")
print(f"{'XGBoost':<25} {auc_xgb:>8.4f} {gini_xgb*100:>7.1f}%")
print("-"*40)
print(f"\nImprovement: +{(gini_xgb - 0.428)*100:.1f}% Gini")

# Verdict
print(f"\nVerdict: ", end = "")
if gini_xgb >=0.75:
    print("Strong Model")
elif gini_xgb >=0.60:
    print("Good Model")
if gini_xgb >=0.40:
    print("Acceptable Model")
else:
    print("Weak Model")

XGBoost Performance:
----------------------------------------
ROC AUC: 0.7192
Gini Coefficient: 0.4384 (43.8%)

Model Comparison: 
----------------------------------------
Model                          AUC     Gini
----------------------------------------
Logistic Regression         0.7140    42.8%
XGBoost                     0.7192    43.8%
----------------------------------------

Improvement: +1.0% Gini

Verdict: Acceptable Model


In [19]:
# Model feature addition
# Investigate sub_grade
print("Sub-grade distribution:")
print(df['sub_grade'].value_counts().sort_index())
print(f"\nUnique Values: {df['sub_grade'].nunique()}")
print(f"Missing Values: {df['sub_grade'].isnull().sum()}")
print(f"\n Default rate by sub_grade sample:")

subgrade_analysis = df.groupby('sub_grade')['default'].agg(
    total = 'count', defaults = 'sum'
).reset_index()

subgrade_analysis['default_rate'] = (
    subgrade_analysis['defaults']/subgrade_analysis['total']*100
).round(1)

print(subgrade_analysis.to_string(index = False))

Sub-grade distribution:
sub_grade
A1    1970
A2    1326
A3    1251
A4    1622
A5    2244
B1    2479
B2    2702
B3    2895
B4    2963
B5    2852
C1    2909
C2    2616
C3    2429
C4    2438
C5    1833
D1    1587
D2    1245
D3    1053
D4     990
D5     807
E1     728
E2     690
E3     580
E4     415
E5     361
F1     261
F2     197
F3     173
F4      99
F5     100
G1      52
G2      62
G3      38
G4      25
G5      14
Name: count, dtype: int64

Unique Values: 35
Missing Values: 0

 Default rate by sub_grade sample:
sub_grade  total  defaults  default_rate
       A1   1970        54           2.7
       A2   1326        52           3.9
       A3   1251        63           5.0
       A4   1622       112           6.9
       A5   2244       163           7.3
       B1   2479       234           9.4
       B2   2702       329          12.2
       B3   2895       400          13.8
       B4   2963       482          16.3
       B5   2852       471          16.5
       C1   2909       582     

### Adding sub_grade - Ordinal Encoding
sub_grade provides 35-level risk segmentation vs 7-level grade.
Default rate spreads from 2.7% (A1) to 76.0% (G4).  
Ordinal encoding preserves risk ordering: A1=0, A2=1...G5=34.  
Note: G4/G5 thin samples (<25 loans) — noisy but XGBoost
handles low-volume splits naturally.

In [20]:
# Create ordinal encoding for sub_grade

sub_grade_order = [ 'A1','A2','A3','A4','A5',
    'B1','B2','B3','B4','B5',
    'C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5',
    'E1','E2','E3','E4','E5',
    'F1','F2','F3','F4','F5',
    'G1','G2','G3','G4','G5'
]
sub_grade_map = {sg: i for i, sg in enumerate(sub_grade_order)}
df['sub_grade_encoded'] = df['sub_grade'].map(sub_grade_map)

print("Sub-grade encoding sample: ")
print(df[['sub_grade','sub_grade_encoded']].drop_duplicates().sort_values(
    'sub_grade_encoded').head(10).to_string(index = False)
)
print(f"\nEncoded range: {df['sub_grade_encoded'].min()}"
    f"to {df['sub_grade_encoded'].max()}"
)
print(f"Missing values: {df['sub_grade_encoded'].isnull().sum()}")

Sub-grade encoding sample: 
sub_grade  sub_grade_encoded
       A1                  0
       A2                  1
       A3                  2
       A4                  3
       A5                  4
       B1                  5
       B2                  6
       B3                  7
       B4                  8
       B5                  9

Encoded range: 0to 34
Missing values: 0


### XGBoost v2 - Sub_grade + Default Parameters
Adding sub_grade_encoded to feature set.  
Establishing new baseline before hyperparameter tuning.

In [21]:
# Enhanced feature set — adding sub_grade_encoded
feature_cols_v2 = [
    'loan_amnt',
    'installment',
    'annual_inc',
    'int_rate',
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc',
    'installment_to_income',
    'fico_range_low',
    'sub_grade_encoded'
]
print(f"Features v1: 14")
print(f"Features v2: {len(feature_cols_v2)}")
print(f"Added: sub_grade_encoded")

# Rebuild Split
X_v2 = df[feature_cols_v2]
y_v2 = df[target_col]

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size = 0.2, 
    random_state = 42, stratify = y_v2
)

# Train model - with default parameters
model_xgb_v2 = XGBClassifier(
    n_estimator = 300, max_depth = 4, learning_rate = 0.05,
    subsample = 0.8, colsample_bytree = 0.8,
    scale_pos_weight = imbalance_ratio,
    reg_alpha = 0.1, reg_lambda = 1.0, random_state = 42,
    eval_metric = 'auc', verbosity = 0
)
model_xgb_v2.fit(X_train_v2,y_train_v2)

# Evaluate
y_pred_v2 = model_xgb_v2.predict_proba(X_test_v2)[:, 1]
auc_v2 = roc_auc_score(y_test_v2, y_pred_v2)
gini_v2 = (2*auc_v2) - 1

print(f"\nXGBoost v2 Performance:")
print("-"*40)
print(f"ROC AUC: {auc_v2:.4f}")
print(f"Gini: {gini_v2*100:.1f}%")
print(f"\nImprovement over v1: +{(gini_v2-0.438)*100:.1f}%")

# Verdict

print(f"\nVerdict: ", end = "")
if gini_v2 >=0.75:
    print("Strong Model")
elif gini_v2 >=0.60:
    print("Good Model")
if gini_v2 >=0.40:
    print("Acceptable Model")
else:
    print("Weak Model")

Features v1: 14
Features v2: 15
Added: sub_grade_encoded



XGBoost v2 Performance:
----------------------------------------
ROC AUC: 0.7174
Gini: 43.5%

Improvement over v1: +-0.3%

Verdict: Acceptable Model


### XGBoost v3 - sub_grade replaces int_rate
int_rate and sub_grade_encoded are highly correlated -
LendingClub assigns rate based on sub_grade.  
Removing int_rate to isolate sub_grade signal.

In [22]:
# v3 — sub_grade replaces int_rate
feature_cols_v3 = [
    'loan_amnt',
    'installment',
    'annual_inc',
    'sub_grade_encoded',  # replaces int_rate
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc',
    'installment_to_income',
    'fico_range_low'
]

print(f"Features v3: {len(feature_cols_v3)}")
print(f"int_rate removed, sub_grade_encoded kept")

X_v3 = df[feature_cols_v3]
y_v3 = df[target_col]

X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y_v3,
    test_size=0.2,
    random_state=42,
    stratify=y_v3
)

model_xgb_v3 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)

model_xgb_v3.fit(X_train_v3, y_train_v3)

y_pred_v3 = model_xgb_v3.predict_proba(X_test_v3)[:, 1]
auc_v3 = roc_auc_score(y_test_v3, y_pred_v3)
gini_v3 = (2 * auc_v3) - 1

print(f"\nXGBoost v3 Performance:")
print("-" * 40)
print(f"ROC AUC:  {auc_v3:.4f}")
print(f"Gini:     {gini_v3*100:.1f}%")
print(f"\nComparison:")
print("-" * 45)
print(f"{'Model':<30} {'Gini':>8}")
print("-" * 45)
print(f"{'XGBoost v1 (no sub_grade)':<30} {'43.8%':>8}")
print(f"{'XGBoost v2 (+sub_grade)':<30} {'43.5%':>8}")
print(f"{'XGBoost v3 (sub_grade only)':<30} {gini_v3*100:>7.1f}%")
print("-" * 45)

Features v3: 14
int_rate removed, sub_grade_encoded kept

XGBoost v3 Performance:
----------------------------------------
ROC AUC:  0.7183
Gini:     43.7%

Comparison:
---------------------------------------------
Model                              Gini
---------------------------------------------
XGBoost v1 (no sub_grade)         43.8%
XGBoost v2 (+sub_grade)           43.5%
XGBoost v3 (sub_grade only)       43.7%
---------------------------------------------


### XGBoost v4 - Hyperparameter Tuning
Feature engineering exhausted - sub_grade and int_rate carry identical signal. Tuning parameters to extract maximum performance from existing feature set.  
Changes: n_estimators 300→500, learning_rate 0.05→0.01, max_depth 4→6.

In [23]:
# v4 — tuned hyperparameters on best feature set (v1)
model_xgb_v4 = XGBClassifier(
    n_estimators = 500,
    max_depth = 6,
    learning_rate = 0.01,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight=imbalance_ratio,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    random_state = 42,
    eval_metric = 'auc',
    verbosity = 0
)

X_v4 = df[feature_cols]
y_v4 = df[target_col]

X_train_v4, X_test_v4, y_train_v4, y_test_v4 = train_test_split(
    X_v4, y_v4,test_size = 0.2,
    random_state = 42, stratify = y_v4
)

print("Training XGBoost v4 — tuned parameters...")
print("n_estimators=500, max_depth=6, learning_rate=0.01")
print("This will take 30-60 seconds...")

model_xgb_v4.fit(X_train_v4, y_train_v4)

y_pred_v4 = model_xgb_v4.predict_proba(X_test_v4)[:, 1]
auc_v4 = roc_auc_score(y_test_v4, y_pred_v4)
gini_v4 = (2 * auc_v4) - 1

print(f"\nXGBoost v4 Performance:")
print("-" * 40)
print(f"ROC AUC: {auc_v4:.4f}")
print(f"Gini: {gini_v4*100:.1f}%")

print(f"\nFull Comparison:")
print("-" * 50)
print(f"{'Model':<35} {'Gini':>8}")
print("-" * 50)
print(f"{'Logistic Regression':<35} {'42.8%':>8}")
print(f"{'XGBoost v1 (default params)':<35} {'43.8%':>8}")
print(f"{'XGBoost v2 (+sub_grade)':<35} {'43.5%':>8}")
print(f"{'XGBoost v3 (sub_grade only)':<35} {'43.7%':>8}")
print(f"{'XGBoost v4 (tuned)':<35} {gini_v4*100:>7.1f}%")
print("-" * 50)

print(f"\nVerdict — ", end = "")
if gini_v4 >= 0.75:
    print("Strong model")
elif gini_v4 >= 0.60:
    print("Good model")
elif gini_v4 >= 0.40:
    print("Acceptable model")
else:
    print("Weak model")

Training XGBoost v4 — tuned parameters...
n_estimators=500, max_depth=6, learning_rate=0.01
This will take 30-60 seconds...

XGBoost v4 Performance:
----------------------------------------
ROC AUC: 0.7180
Gini: 43.6%

Full Comparison:
--------------------------------------------------
Model                                   Gini
--------------------------------------------------
Logistic Regression                    42.8%
XGBoost v1 (default params)            43.8%
XGBoost v2 (+sub_grade)                43.5%
XGBoost v3 (sub_grade only)            43.7%
XGBoost v4 (tuned)                     43.6%
--------------------------------------------------

Verdict — Acceptable model


### XGBoost Conclusion

All four XGBoost variants converged within 1% Gini of the 
logistic regression baseline (42.8%). 

Key finding: The dataset has a signal ceiling around 43-44% 
Gini driven by:
1. Feature correlation - loan_amnt and installment encode 
   similar information
2. Linear relationships - grade and int_rate have largely 
   linear default gradients
3. Missing behavioural data - payment patterns and account 
   activity would unlock non-linear signal

Production recommendation: Logistic regression remains the 
appropriate choice - equivalent performance, superior 
interpretability, native Basel III compliance.

To break the 50% Gini ceiling: gradient boosting with 
bureau-level behavioural data and SMOTE resampling.